# Marketing Campaign Data Analytics Project

## Customer Personality Analysis Using Python

### Project Overview

This project performs data analytics on a marketing campaign dataset
containing customer demographic information, purchasing behaviour,
campaign responses, and channel activity.

The main objectives are:

1. Understand the structure of the customer dataset.
2. Clean and preprocess the data.
3. Perform exploratory data analysis.
4. Calculate important business KPIs.
5. Analyse customer spending behaviour.
6. Analyse purchasing channels.
7. Analyse marketing campaign performance.
8. Study customer demographics and their relationship with spending.
9. Segment customers based on their behaviour.
10. Generate meaningful business insights.
11. Export cleaned data and analytical results.

Tools used:

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-learn
- Jupyter Notebook

## 1. Import Required Libraries

The following Python libraries are used for data loading,
cleaning, analysis, visualization, and customer segmentation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')


## 2. Create Project Folders

The project will use separate folders for:

- Data
- Outputs
- Figures

This keeps the project organized.

In [ ]:
print('Project folder structure is ready.')


## 3. Load the Dataset

The uploaded marketing campaign dataset is semicolon-separated.

Therefore, `sep=";"` is required while reading the CSV file.

In [ ]:
from pathlib import Path

# Support both the recommended project structure and a local notebook run.
project_root = Path('..') if Path('../data').exists() else Path('.')
project_root.mkdir(parents=True, exist_ok=True)
(project_root / 'data').mkdir(exist_ok=True)
(project_root / 'outputs').mkdir(exist_ok=True)
(project_root / 'figures').mkdir(exist_ok=True)

data_candidates = [
    project_root / 'data' / 'marketing_campaign.csv',
    Path('marketing_campaign.csv'),
    Path('marketing_campaign(1).csv'),
    Path('../marketing_campaign.csv'),
]

data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        'marketing_campaign.csv was not found. Place the dataset in the data folder or notebook folder.'
    )

df = pd.read_csv(data_path, sep=';')

print(f'Dataset loaded successfully from: {data_path}')


In [ ]:
df.head()

In [ ]:
print(df.shape)

## 4. Dataset Information

The `info()` function provides information about:

- Column names
- Number of non-null values
- Data types
- Memory usage

In [ ]:
df.info()

## 5. Statistical Summary

The statistical summary helps us understand numerical variables
such as income, spending, purchases, and campaign responses.

In [ ]:
display(df.describe())

## 6. Missing Value Analysis

Missing values can affect analysis and machine learning.

We check every column to identify missing data.

In [ ]:
missing_values = df.isnull().sum()

missing_table = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage": (missing_values / len(df)) * 100
})

display(missing_table[missing_table["Missing Values"] > 0])

## 7. Duplicate Record Analysis

Duplicate records can lead to incorrect analysis.

We check whether duplicate customer records exist.

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

print("Dataset shape after removing duplicates:", df.shape)

## 8. Standardize Column Names

Column names are converted into a consistent format
to make them easier to work with.

In [ ]:
df.columns = df.columns.str.strip()

print(df.columns.tolist())

## 9. Convert Customer Registration Date

`Dt_Customer` contains the date on which the customer joined.

We convert this column from text into a proper datetime format.

In [ ]:
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], errors="coerce")

print(df["Dt_Customer"].dtype)

## 10. Handle Missing Income Values

The dataset contains missing values in the `Income` column.

The median income is used to fill missing values because
the median is less affected by extreme values than the mean.

In [ ]:
median_income = df["Income"].median()

df["Income"] = df["Income"].fillna(median_income)

print("Remaining missing Income values:", df["Income"].isnull().sum())

## 11. Handle Invalid Birth Years

Some records contain unrealistic birth years.

These records can create unrealistic customer ages.

We identify customers born before 1920 and remove those records
from the analytical dataset.

In [ ]:
print("Records with Year_Birth before 1920:")

display(
    df[df["Year_Birth"] < 1920][
        ["ID", "Year_Birth", "Income"]
    ]
)

In [ ]:
df = df[df["Year_Birth"] >= 1920].copy()

print("Dataset shape after removing invalid birth years:", df.shape)

## 12. Check Extreme Income Values

We inspect very high income values.

Extreme values are important because they can strongly influence
averages and visualizations.

In [ ]:
display(df[df["Income"] > 200000][["ID", "Income"]])

In [ ]:
df = df[df["Income"] <= 200000].copy()

print("Dataset shape after handling extreme income:", df.shape)

## 13. Feature Engineering

We create additional variables from the existing dataset.

The maximum customer registration year is used as the reference year
for calculating approximate customer age.

In [ ]:
reference_year = df["Dt_Customer"].dt.year.max()

df["Age"] = reference_year - df["Year_Birth"]

print("Reference Year:", reference_year)

display(df[["Year_Birth", "Age"]].head())

## 14. Calculate Total Customer Spending

The dataset contains spending on:

- Wine
- Fruits
- Meat
- Fish
- Sweets
- Gold products

We combine these variables to calculate total spending.

In [ ]:
spending_columns = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds"
]

df["Total_Spending"] = df[spending_columns].sum(axis=1)

display(df[["ID", "Total_Spending"]].head())

## 15. Calculate Total Purchases

Customers can purchase products through:

- Web
- Catalog
- Store

We calculate the total number of purchases across these channels.

In [ ]:
purchase_columns = [
    "NumWebPurchases",
    "NumCatalogPurchases",
    "NumStorePurchases"
]

df["Total_Purchases"] = df[purchase_columns].sum(axis=1)

display(df[["ID", "Total_Purchases"]].head())

## 16. Calculate Campaign Acceptance

The dataset contains five previous campaign response variables:

- AcceptedCmp1
- AcceptedCmp2
- AcceptedCmp3
- AcceptedCmp4
- AcceptedCmp5

We calculate the total number of previous campaigns accepted
by each customer.

In [ ]:
campaign_columns = [
    "AcceptedCmp1",
    "AcceptedCmp2",
    "AcceptedCmp3",
    "AcceptedCmp4",
    "AcceptedCmp5"
]

df["Total_Campaign_Accepted"] = df[campaign_columns].sum(axis=1)

display(
    df[["ID", "Total_Campaign_Accepted", "Response"]].head()
)

## 17. Calculate Total Children

We combine children and teenagers living in the household
to create a total household children variable.

In [ ]:
df["Total_Children"] = df["Kidhome"] + df["Teenhome"]

display(
    df[["Kidhome", "Teenhome", "Total_Children"]].head()
)

## 18. Key Performance Indicators

Because this dataset does not contain impressions, clicks, advertising cost,
or revenue, traditional KPIs such as CTR and ROI cannot be calculated directly.

Instead, we calculate valid KPIs from the available data:

- Total Customers
- Average Income
- Average Spending
- Total Spending
- Average Purchases
- Campaign Response Rate
- Campaign Acceptance Rate
- Complaint Rate
- Average Recency

In [ ]:
total_customers = len(df)

average_income = df["Income"].mean()
average_spending = df["Total_Spending"].mean()
total_spending = df["Total_Spending"].sum()
average_purchases = df["Total_Purchases"].mean()

response_rate = df["Response"].mean() * 100

campaign_acceptance_rate = (
    df["Total_Campaign_Accepted"].sum()
    / (len(df) * 5)
) * 100

complaint_rate = df["Complain"].mean() * 100

average_recency = df["Recency"].mean()

kpi_table = pd.DataFrame({
    "KPI": [
        "Total Customers",
        "Average Income",
        "Average Spending",
        "Total Spending",
        "Average Purchases",
        "Campaign Response Rate",
        "Campaign Acceptance Rate",
        "Complaint Rate",
        "Average Recency"
    ],
    "Value": [
        total_customers,
        average_income,
        average_spending,
        total_spending,
        average_purchases,
        response_rate,
        campaign_acceptance_rate,
        complaint_rate,
        average_recency
    ]
})

display(kpi_table)

## 19. Customer Response Analysis

The `Response` column represents the customer's response
to the most recent campaign.

- 0 = Did not respond
- 1 = Responded

In [ ]:
response_counts = df["Response"].value_counts()

display(response_counts)

In [ ]:
response_counts.sort_index().plot(kind='bar', figsize=(7, 5))

plt.title('Customer Response to Latest Campaign')
plt.xlabel('Response')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 20. Customer Education Analysis

We analyse the educational background of customers
to understand the customer population.

In [ ]:
education_counts = df["Education"].value_counts()

display(education_counts)

In [ ]:
education_counts.sort_values(ascending=False).plot(kind='bar', figsize=(8, 5))

plt.title('Customer Distribution by Education')
plt.xlabel('Education')
plt.ylabel('Number of Customers')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 21. Marital Status Analysis

We analyse the distribution of marital-status categories and compare the latest campaign response rate across those groups.

The dataset contains categories such as Single, Married, Together, Divorced, Widow, and a small number of unusual categories. These categories are retained rather than silently removed so that the original data is not altered beyond the documented cleaning steps.


In [ ]:
marital_counts = df['Marital_Status'].value_counts()

marital_response_rate = (
    df.groupby('Marital_Status')['Response']
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

marital_summary = pd.DataFrame({
    'Customer_Count': marital_counts,
    'Response_Rate_%': marital_response_rate
}).sort_values('Customer_Count', ascending=False)

display(marital_summary)


In [ ]:
marital_counts.plot(kind='bar', figsize=(9, 5))

plt.title('Customer Distribution by Marital Status')
plt.xlabel('Marital Status')
plt.ylabel('Number of Customers')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 22. Age Distribution

Age is analysed to understand the demographic structure
of the customer base.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df['Age'].dropna(), bins=25)
plt.title('Customer Age Distribution')
plt.xlabel('Age')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()


## 23. Income Distribution

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df['Income'].dropna(), bins=30)
plt.title('Customer Income Distribution')
plt.xlabel('Income')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()


## 24. Customer Spending Analysis

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df['Total_Spending'].dropna(), bins=30)
plt.title('Distribution of Total Customer Spending')
plt.xlabel('Total Spending')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()


## 25. Product Category Analysis

We compare customer spending across different product categories.

In [ ]:
product_spending = df[spending_columns].sum().sort_values(ascending=False)

display(product_spending)

In [ ]:
plt.figure(figsize=(9, 5))

product_spending.plot(kind="bar")

plt.title("Total Spending by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Total Spending")

plt.xticks(rotation=30)

plt.show()

## 26. Purchase Channel Analysis

Customers make purchases through:

- Web
- Catalog
- Store

These are purchase channels in this dataset. They should not be described as advertising channels such as Google Ads or Instagram because those fields are not present in the source data.


In [ ]:
channel_purchases = df[
    ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]
].sum().sort_values(ascending=False)

display(channel_purchases)

In [ ]:
plt.figure(figsize=(8, 5))

channel_purchases.plot(kind="bar")

plt.title("Purchases by Channel")
plt.xlabel("Purchase Channel")
plt.ylabel("Number of Purchases")

plt.xticks(rotation=20)

plt.show()

## 27. Web Activity Analysis

We analyse the relationship between monthly web visits and web purchases.

This is a customer web-activity measure, not a true advertising click-through or conversion rate, because the dataset does not provide ad impressions, ad clicks, or a defined conversion event.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['NumWebVisitsMonth'], df['NumWebPurchases'], alpha=0.7)
plt.title('Web Visits vs Web Purchases')
plt.xlabel('Web Visits per Month')
plt.ylabel('Web Purchases')
plt.tight_layout()
plt.show()


## 28. Marketing Campaign Analysis

The dataset contains five previous campaigns.

We calculate the number of customers who accepted each campaign.

In [ ]:
campaign_acceptance = df[campaign_columns].sum().sort_values(
    ascending=False
)

display(campaign_acceptance)

In [ ]:
plt.figure(figsize=(8, 5))

campaign_acceptance.plot(kind="bar")

plt.title("Customers Accepting Previous Campaigns")
plt.xlabel("Campaign")
plt.ylabel("Number of Customers")

plt.xticks(rotation=0)

plt.show()

## 29. Campaign Response by Education

We compare the latest campaign response rate across
different education groups.

In [ ]:
response_by_education = (
    df.groupby("Education")["Response"]
    .mean()
    .sort_values(ascending=False)
    * 100
)

display(response_by_education)

In [ ]:
plt.figure(figsize=(8, 5))

response_by_education.plot(kind="bar")

plt.title("Campaign Response Rate by Education")
plt.xlabel("Education")
plt.ylabel("Response Rate (%)")

plt.xticks(rotation=30)

plt.show()

## 30. Campaign Response by Marital Status

In [ ]:
response_by_marital = (
    df.groupby("Marital_Status")["Response"]
    .mean()
    .sort_values(ascending=False)
    * 100
)

display(response_by_marital)

In [ ]:
plt.figure(figsize=(9, 5))

response_by_marital.plot(kind="bar")

plt.title("Campaign Response Rate by Marital Status")
plt.xlabel("Marital Status")
plt.ylabel("Response Rate (%)")

plt.xticks(rotation=30)

plt.show()

## 31. Spending by Education

We compare average customer spending across education groups.

In [ ]:
spending_by_education = (
    df.groupby("Education")["Total_Spending"]
    .mean()
    .sort_values(ascending=False)
)

display(spending_by_education)

In [ ]:
plt.figure(figsize=(8, 5))

spending_by_education.plot(kind="bar")

plt.title("Average Spending by Education")
plt.xlabel("Education")
plt.ylabel("Average Spending")

plt.xticks(rotation=30)

plt.show()

## 32. Income and Spending Relationship

We investigate whether customers with higher income
tend to have higher total spending.

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(df['Income'], df['Total_Spending'], c=df['Response'], alpha=0.7)
plt.title('Income vs Total Spending')
plt.xlabel('Income')
plt.ylabel('Total Spending')
plt.tight_layout()
plt.show()


## 33. Correlation Analysis

Correlation helps identify relationships between numerical variables.

A positive correlation means two variables tend to increase together,
while a negative correlation means one tends to decrease as the other increases.

In [ ]:
correlation_columns = [
    "Income",
    "Age",
    "Recency",
    "Total_Spending",
    "Total_Purchases",
    "NumWebVisitsMonth",
    "Total_Campaign_Accepted",
    "Response"
]

correlation_matrix = df[correlation_columns].corr()

display(correlation_matrix)

In [ ]:
plt.figure(figsize=(10, 7))
im = plt.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, label='Correlation')
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=45, ha='right')
plt.yticks(range(len(correlation_matrix.index)), correlation_matrix.index)
for row in range(correlation_matrix.shape[0]):
    for col in range(correlation_matrix.shape[1]):
        plt.text(col, row, f"{correlation_matrix.iloc[row, col]:.2f}", ha='center', va='center')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()


## 34. Top Customers by Total Spending

We identify customers with the highest total spending.

In [ ]:
top_customers = (
    df[
        [
            "ID",
            "Income",
            "Age",
            "Education",
            "Marital_Status",
            "Total_Spending",
            "Total_Purchases",
            "Response"
        ]
    ]
    .sort_values("Total_Spending", ascending=False)
    .head(10)
)

display(top_customers)

## 35. Customer Segmentation Using K-Means

Customer segmentation divides customers into groups with similar characteristics.

We use:

- Income
- Recency
- Total Spending
- Web Purchases
- Catalog Purchases
- Store Purchases
- Campaign Acceptance

`Total_Purchases` is not included together with its three component purchase-channel variables because that would give purchase volume extra weight during clustering.

K-Means clustering is used to identify customer segments.


In [ ]:
segmentation_columns = [
    'Income',
    'Recency',
    'Total_Spending',
    'NumWebPurchases',
    'NumCatalogPurchases',
    'NumStorePurchases',
    'Total_Campaign_Accepted'
]

X = df[segmentation_columns].copy()

print('Segmentation feature matrix shape:', X.shape)


## 36. Feature Scaling

The variables have different scales.

For example:

- Income can be tens of thousands.
- Recency can be below 100.
- Campaign acceptance can be between 0 and 5.

StandardScaler puts the variables on a comparable scale.

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Scaling completed.")

## 37. Selecting the Number of Clusters

The Elbow Method is used to examine different values of K.

The within-cluster sum of squares is represented by inertia.

In [ ]:
inertia = []

K_values = range(2, 9)

for k in K_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    kmeans.fit(X_scaled)
    
    inertia.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(K_values, inertia, marker="o")

plt.title("Elbow Method for K-Means")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")

plt.show()

## 38. Silhouette Score

The silhouette score measures how well customers fit
within their assigned clusters.

A higher score indicates better separation between clusters.

In [ ]:
silhouette_scores = {}

for k in range(2, 9):
    
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = kmeans.fit_predict(X_scaled)
    
    score = silhouette_score(X_scaled, labels)
    
    silhouette_scores[k] = score

display(pd.DataFrame(
    list(silhouette_scores.items()),
    columns=["Number_of_Clusters", "Silhouette_Score"]
))

## 39. Select the Cluster Number

The elbow plot is used for visual inspection of the candidate range, while the silhouette score is used by the code to select the final K value.

The code below selects the K value with the highest silhouette score among the tested values.


In [ ]:
best_k = max(
    silhouette_scores,
    key=silhouette_scores.get
)

print("Selected number of clusters:", best_k)

## 40. Final Customer Segmentation Model

In [ ]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["Customer_Segment"] = kmeans.fit_predict(X_scaled)

print("Customer segmentation completed.")

## 41. Customer Segment Profiles

We calculate the average behaviour of customers
within each segment.

In [ ]:
segment_profile = (
    df.groupby('Customer_Segment')[segmentation_columns]
    .mean()
    .round(2)
)

display(segment_profile)


## 42. Customer Segment Distribution

In [ ]:
segment_counts = df["Customer_Segment"].value_counts().sort_index()

display(segment_counts)

In [ ]:
segment_counts.plot(kind='bar', figsize=(8, 5))
plt.title('Customer Distribution by Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 43. Spending Across Customer Segments

In [ ]:
segment_spending = (
    df.groupby("Customer_Segment")["Total_Spending"]
    .mean()
    .sort_values(ascending=False)
)

display(segment_spending)

In [ ]:
plt.figure(figsize=(8, 5))

segment_spending.plot(kind="bar")

plt.title("Average Spending by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Average Spending")

plt.show()

## 44. Campaign Response by Customer Segment

In [ ]:
segment_response = (
    df.groupby("Customer_Segment")["Response"]
    .mean()
    * 100
)

display(segment_response)

In [ ]:
plt.figure(figsize=(8, 5))

segment_response.plot(kind="bar")

plt.title("Campaign Response Rate by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Response Rate (%)")

plt.show()

## 45. Final Dataset Summary

After cleaning and feature engineering, we inspect
the final dataset.

In [ ]:
print("Final Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

display(df.head())

## 46. Export Cleaned Dataset

The cleaned dataset contains the original variables
plus the newly engineered analytical features.

The file is saved for future use in dashboards and reports.

In [ ]:
df.to_csv(
    project_root / 'outputs' / 'cleaned_marketing_data.csv',
    index=False
)

print('Cleaned dataset saved successfully.')


## 47. Export KPI Results

In [ ]:
kpi_table.to_csv(
    project_root / 'outputs' / 'marketing_kpis.csv',
    index=False
)

print('KPI table saved successfully.')


## 48. Export Analytical Results

Important analytical tables are saved separately so they can
later be used in the project report and dashboard.

In [ ]:
product_spending.to_csv(
    project_root / 'outputs' / 'product_spending.csv'
)

channel_purchases.to_csv(
    project_root / 'outputs' / 'channel_purchases.csv'
)

campaign_acceptance.to_csv(
    project_root / 'outputs' / 'campaign_acceptance.csv'
)

marital_summary.to_csv(
    project_root / 'outputs' / 'marital_analysis.csv'
)

response_by_education.to_csv(
    project_root / 'outputs' / 'response_by_education.csv'
)

spending_by_education.to_csv(
    project_root / 'outputs' / 'spending_by_education.csv'
)

segment_profile.to_csv(
    project_root / 'outputs' / 'customer_segments.csv'
)

print('Analytical results exported successfully.')


In [ ]:
# Quantitative business insights generated directly from the analysed data.
top_product = product_spending.index[0]
top_product_value = product_spending.iloc[0]
top_channel = channel_purchases.index[0]
top_channel_value = channel_purchases.iloc[0]
top_campaign = campaign_acceptance.index[0]
top_campaign_value = campaign_acceptance.iloc[0]
top_segment = segment_spending.index[0]
top_segment_spending = segment_spending.iloc[0]
best_response_segment = segment_response.idxmax()
best_response_segment_rate = segment_response.max()

print(f'Highest-spending product category: {top_product} ({top_product_value:,.0f} total spending)')
print(f'Most-used purchase channel: {top_channel} ({top_channel_value:,.0f} purchases)')
print(f'Most accepted previous campaign: {top_campaign} ({top_campaign_value:,.0f} customers)')
print(f'Highest-spending customer segment: Segment {top_segment} (average spending {top_segment_spending:,.2f})')
print(f'Highest latest-campaign response segment: Segment {best_response_segment} ({best_response_segment_rate:.2f}%)')
print(f'Overall latest-campaign response rate: {response_rate:.2f}%')
print(f'Overall previous-campaign acceptance rate: {campaign_acceptance_rate:.2f}%')


## 49. Business Insights

The analysis provides several important insights:

1. Customer spending varies significantly across customers.

2. Product categories contribute differently to total customer spending.

3. Customers use different purchasing channels, including web,
   catalog, and physical stores.

4. The latest campaign response rate can be measured directly
   from the Response variable.

5. Previous campaign acceptance can be compared using
   AcceptedCmp1 to AcceptedCmp5.

6. Customer demographics such as education and marital status
   can be compared with spending and campaign response.

7. Income and total spending can be analysed to understand
   customer purchasing behaviour.

8. Customer segmentation using K-Means groups customers
   according to their income, recency, spending, purchasing
   behaviour, and campaign acceptance.

9. These customer segments can be used to support targeted
   marketing strategies.

10. The dataset does not contain advertising impressions,
    clicks, advertising cost, or revenue. Therefore, CTR,
    advertising conversion rate, and ROI cannot be calculated
    directly from this dataset.

## 50. Conclusion

This project analysed customer marketing and purchasing data
using Python-based data analytics techniques.

The project began with data loading and preprocessing.
Missing values, duplicate records, invalid birth years,
and extreme income values were handled.

Several new features were created, including:

- Age
- Total Spending
- Total Purchases
- Total Campaign Acceptance
- Total Children

Exploratory data analysis was then performed to understand:

- Customer demographics
- Income distribution
- Spending behaviour
- Product category preferences
- Purchasing channels
- Campaign acceptance
- Campaign response
- Relationships between numerical variables

Customer segmentation was also performed using K-Means clustering.
This helped divide customers into groups with similar purchasing
and marketing behaviour.

The analysis demonstrates how data analytics can transform raw
customer information into meaningful insights that can support
marketing analysis and customer targeting.

Because the dataset does not contain advertising cost, impressions,
clicks, or revenue, metrics such as CTR and ROI were not artificially
calculated. Instead, KPIs based on the available customer,
purchase, and campaign-response information were used.

## 51. Project Output Files

The following files have been generated for the project.

In [ ]:
print('Files generated in the outputs folder:')

for file in sorted((project_root / 'outputs').iterdir()):
    if file.is_file():
        print('-', file.name)
